In [1]:
import pluma.schema.outdoor as outdoor

# From utils
import schemas.maat_schema as maat
import schemas.missing_sync as missing_sync
# Pluma-python API
from modules import *

/Users/raquel/micromamba/envs/tese/lib/python3.9/site-packages/heartpy/datautils.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


In [2]:
from utils import *
import utils.for_empatica as empatica
import utils.for_climate as climate

print(sourcedata)
os.listdir(os.path.join(sourcedata, "data"))

Current city is copenhagen!
If you wish to change the city, please edit the value in the __init__.py file
/Users/raquel/Desktop/uni BA/4th semester/master-thesis/sourcedata


['OE009',
 'OE007',
 '.DS_Store',
 'OE022',
 'OE012',
 'OE015',
 'OE023',
 'OE024',
 'OE002',
 'OE005',
 'OE004',
 'OE021',
 'OE019',
 'OE010',
 'OE017',
 'OE011',
 'OE018',
 'OE020']

In [3]:
os.listdir(os.path.join(sourcedata, "data", "OE009"))

['Copenhagen_Nordhavn_sub-OE203009_2024-06-25T075820Z',
 'Copenhagen_Norrebro_sub-OE201009_2024-06-24T134202Z']

In [4]:
# Import necessary modules
from utils import *
import utils.for_empatica as empatica
import utils.for_climate as climate

# Pluma-python API  
from modules import *
import pluma.schema.outdoor as outdoor
import schemas.maat_schema as maat

# ---------------------------------------------------------------------
# CONFIGURATION: Specify one participant and one session to process.
# Replace the values below with the appropriate folder names.
participant_folder = "OE009"  
session_folder     = "Copenhagen_Nordhavn_sub-OE203009_2024-06-25T075820Z"     

# ---------------------------------------------------------------------
# MAIN SCRIPT FOR A SINGLE DATASET

# Set the data directory
datadir = os.path.join(sourcedata, 'data')

# Build the full paths for the participant and session
participant_path = os.path.join(datadir, participant_folder)
session_path = os.path.join(participant_path, session_folder)

if not os.path.isdir(session_path):
    raise FileNotFoundError(f"Session path {session_path} does not exist.")

# Extract the session name using your helper function.
session_name = extract_session_name(session_folder)

# Create output directories if they do not exist.
# Geodata output directory:
output = os.path.join(sourcedata, 'supp', 'geodata', 'log', f'sub-{participant_folder}', f'ses-{session_name}')
if not os.path.exists(output): 
    os.makedirs(output)

# Empatica output directory:
csv_outdir = os.path.join(sourcedata, 'supp', 'stress_csv', f'sub-{participant_folder}', f'ses-{session_name}')
if not os.path.exists(csv_outdir): 
    os.makedirs(csv_outdir)

# Try to load the dataset using different schemas.
if "Maat" in session_folder:
    print("This is a Maat acquisition...")
    calibrate_ubx_to_harp = True # When true there is better synchronization
    ubx = True                   # False only for VR acquisitions    
    dataset    = load_dataset(session_path, schema=maat.build_schema, ubx=ubx, calibrate_ubx_to_harp=calibrate_ubx_to_harp)    
    plot_summary(dataset, plot_sync_lookup=ubx and calibrate_ubx_to_harp)
else:    
    try:
        print("Calibrating UBX to Harp...")
        calibrate_ubx_to_harp = True # When true there is better synchronization
        ubx = True                   # False only for VR acquisitions
        dataset    = load_dataset(session_path, schema=outdoor.build_schema, ubx=ubx, unity=False, calibrate_ubx_to_harp=calibrate_ubx_to_harp)
        plot_summary(dataset, plot_sync_lookup=ubx and calibrate_ubx_to_harp)
    except:
        try:
            print("Trying to load by the missing GPS synchronization schema...")
            calibrate_ubx_to_harp = False # When true there is better synchronization
            ubx = True                    # False only for VR acquisitions
            dataset    = load_dataset(session_path, ubx=ubx, unity=False, calibrate_ubx_to_harp=calibrate_ubx_to_harp, schema=missing_sync.build_schema)
        except Exception as e:
            print(f"Error in {participant_folder}-{session_name}")
            raise e  # Or handle the error as needed

# Save geodata to CSV.
climate.geodata_to_csv(dataset, participant_folder, session_name, output)

# Save Empatica (and ECG) data to CSV.
try:
    empatica.empatica_and_ecg_to_csv(dataset, csv_outdir)
    print(f"Successfully saved data for {participant_folder}-{session_name}!")
except Exception as e:
    print(f"Error in {participant_folder}-{session_name}")
    print("No empatica data found...")

Calibrating UBX to Harp...
Trying to load by the missing GPS synchronization schema...
Error in OE009-Nordhavn


/Users/raquel/micromamba/envs/tese/lib/python3.9/site-packages/pluma/io/harp.py:43: UserWarning: Harp stream file            @(WIN) --> \Users\raquel\Desktop\uni BA\4th semester\master-thesis\sourcedata\data\OE009\Copenhagen_Nordhavn_sub-OE203009_2024-06-25T075820Z\Streams_32 could not be found.
  warnings.warn(f"Harp stream file\
/Users/raquel/micromamba/envs/tese/lib/python3.9/site-packages/pluma/io/harp.py:43: UserWarning: Harp stream file            @(WIN) --> \Users\raquel\Desktop\uni BA\4th semester\master-thesis\sourcedata\data\OE009\Copenhagen_Nordhavn_sub-OE203009_2024-06-25T075820Z\Streams_33 could not be found.
  warnings.warn(f"Harp stream file\
/Users/raquel/micromamba/envs/tese/lib/python3.9/site-packages/pluma/io/harp.py:43: UserWarning: Harp stream file            @(WIN) --> \Users\raquel\Desktop\uni BA\4th semester\master-thesis\sourcedata\data\OE009\Copenhagen_Nordhavn_sub-OE203009_2024-06-25T075820Z\Streams_35 could not be found.
  warnings.warn(f"Harp stream file\
/

AttributeError: 'DataFrame' object has no attribute 'Value0'

In [1]:
import shapely
import geopandas

print(shapely.__version__)
print(geopandas.__version__)

1.8.5.post1
0.12.2


In [2]:
from utils import *
import utils.for_empatica as empatica
import utils.for_climate as climate

# ---------------------------------------------------------------------
# MAIN SCRIPT
datadir    = os.path.join(sourcedata, 'data')
for participant_folder in os.listdir(datadir):
    if participant_folder.startswith("OE"):
        participant_path = os.path.join(datadir, participant_folder)
        for session_folder in os.listdir(participant_path):
            session_path = os.path.join(participant_path, session_folder)
            if os.path.isdir(session_path):
                session_name = extract_session_name(session_folder)
                # Geodata output
                output = os.path.join(sourcedata, 'supp', 'geodata', 'log', f'sub-{participant_folder}', f'ses-{session_name}')
                if not os.path.exists(output): 
                    os.makedirs(output)
                # Empatica output
                csv_outdir = os.path.join(sourcedata, 'supp', 'stress_csv', f'sub-{participant_folder}', f'ses-{session_name}')
                if not os.path.exists(csv_outdir): 
                    os.makedirs(csv_outdir)
                # Try to load the dataset using different schemas.
                if "Maat" in session_folder:
                    print("This is a Maat acquisition...")
                    calibrate_ubx_to_harp = True # When true there is better synchronization
                    ubx = True                   # False only for VR acquisitions    
                    dataset    = load_dataset(session_path, schema=maat.build_schema, ubx=ubx, calibrate_ubx_to_harp=calibrate_ubx_to_harp)    
                else:    
                    try:
                        calibrate_ubx_to_harp = True # When true there is better synchronization
                        ubx = True                   # False only for VR acquisitions
                        dataset    = load_dataset(session_path, schema=outdoor.build_schema, ubx=ubx, unity=False, calibrate_ubx_to_harp=calibrate_ubx_to_harp)
                        plot_summary(dataset, plot_sync_lookup=ubx and calibrate_ubx_to_harp)
                    except:
                        try:
                            print("Trying to load by the missing GPS synchronization schema...")
                            calibrate_ubx_to_harp = False # When true there is better synchronization
                            ubx = True                    # False only for VR acquisitions
                            dataset    = load_dataset(session_path, ubx=ubx, unity=False, calibrate_ubx_to_harp=calibrate_ubx_to_harp, schema=missing_sync.build_schema)
                        except Exception as e:
                            print(f"Error in {participant_folder}-{session_name}")
                            raise e  # Or handle the error as needed
                # Save geodata to csv
                climate.geodata_to_csv(dataset, participant_folder, session_name, output)
                # Save empatica data to csv
                try:
                    empatica.empatica_and_ecg_to_csv(dataset, csv_outdir) 
                    print(f'Successfully saved data for {participant_folder}-{session_name}!')
                except:
                    print(f"Error in {participant_folder}-{session_name}")
                    print("No empatica data found...")
                    continue

Current city is copenhagen!
If you wish to change the city, please edit the value in the __init__.py file
Trying to load by the missing GPS synchronization schema...
Error in OE009-Nordhavn


NameError: name 'load_dataset' is not defined